In [1]:
import pinecone
import os
from langchain.vectorstores import Pinecone
from langchain.embeddings.openai import OpenAIEmbeddings
import time
from dotenv import load_dotenv

load_dotenv()

pinecone.init()

/opt/homebrew/lib/python3.11/site-packages/pinecone/index.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
/opt/homebrew/lib/python3.11/site-packages/deeplake/util/check_latest_version.py:32: UserWarning: A newer version of deeplake (3.6.12) is available. It's recommended that you update to the latest version using `pip install -U deeplake`.
  warnings.warn(


In [2]:
pinecone_db = Pinecone.from_existing_index(
            "aida", embedding=OpenAIEmbeddings(), namespace="functions_test"
    )

In [3]:
import json
import openai
import requests
from tenacity import retry, wait_random_exponential, stop_after_attempt
from termcolor import colored
import asyncio
import httpx
# from memory import MemoryManager

GPT_MODEL = "gpt-3.5-turbo-0613"
GPT4_MODEL = "gpt-4-0613"

@retry(wait=wait_random_exponential(min=1, max=40), stop=stop_after_attempt(3))
def chat_completion_request(messages, functions=None, function_call=None, model=GPT4_MODEL):
    headers = {
        "Content-Type": "application/json",
        "Authorization": "Bearer " + openai.api_key,
    }
    json_data = {"model": model, "messages": messages}
    if functions is not None:
        json_data.update({"functions": functions})
    if function_call is not None:
        json_data.update({"function_call": function_call})
    try:
        response = requests.post(
            "https://api.openai.com/v1/chat/completions",
            headers=headers,
            json=json_data,
        )
        return response
    except Exception as e:
        print("Unable to generate ChatCompletion response")
        print(f"Exception: {e}")
        return e

def pretty_print_conversation(messages):
    role_to_color = {
        "system": "red",
        "user": "green",
        "assistant": "blue",
        "function": "magenta",
    }
    formatted_messages = []
    for message in messages:
        if message["role"] == "system":
            formatted_messages.append(f"system: {message['content']}\n")
        elif message["role"] == "user":
            formatted_messages.append(f"user: {message['content']}\n")
        elif message["role"] == "assistant" and message.get("function_call"):
            formatted_messages.append(f"assistant: {message['function_call']}\n")
        elif message["role"] == "assistant" and not message.get("function_call"):
            formatted_messages.append(f"assistant: {message['content']}\n")
        elif message["role"] == "function":
            formatted_messages.append(f"function ({message['name']}): {message['content']}\n")
    for formatted_message in formatted_messages:
        print(
            colored(
                formatted_message,
                role_to_color[messages[formatted_messages.index(formatted_message)]["role"]],
            )
        )

# #leaving the async function here to make it easier for building large tests
# async def test_getFunctions(agent_response):
#     if agent_response is None:
#         return
#     if agent_response.json()["choices"][0]["message"]["function_call"] is None:
#         return
#     function_call = agent_response.json()["choices"][0]["message"]["function_call"]
#     print(function_call['arguments'])
#     data = {
#         'categories': ".",
#         'actions': q,
#         'num_results': 2,
#         'similarity_threshold': 0.7 
#     }
#     async with httpx.AsyncClient() as client:  # using httpx AsyncClient
#         response = await client.post("http://127.0.0.1:8000/get_functions/", data=data)  # async post request
#     print(response.status_code)

#     with open('tests/response_agent.json', 'a') as f:
#         f.write('\n')
#         json.dump(response.json(), f)

#     asyncio.run(test_getFunctions())

functions = [
{'name': 'query_plan_tool',
 'description': '        This is a query plan tool that takes in a list of tools and executes a query plan over these tools to answer a query. The query plan is a DAG of query nodes.\n\nGiven a list of tool names and the query plan schema, you can choose to generate a query plan to answer a question.\n\nThe tool names and descriptions are as follows:\n\n\n\n        Tool Name: sept_2022\nTool Description: Provides information about Uber quarterly financials ending September 2022 \n\nTool Name: june_2022\nTool Description: Provides information about Uber quarterly financials ending June 2022 \n\nTool Name: march_2022\nTool Description: Provides information about Uber quarterly financials ending March 2022 \n        ',
 'parameters': {'title': 'QueryPlan',
  'description': "Query plan.\n\nContains a list of QueryNode objects (which is a recursive object).\nOut of the list of QueryNode objects, one of them must be the root node.\nThe root node is the one that isn't a dependency of any other node.",
  'type': 'object',
  'properties': {'nodes': {'title': 'Nodes',
    'description': 'The original question we are asking.',
    'type': 'array',
    'items': {'$ref': '#/definitions/QueryNode'}}},
  'required': ['nodes'],
  'definitions': {'QueryNode': {'title': 'QueryNode',
    'description': 'Query node.\n\nA query node represents a query (query_str) that must be answered.\nIt can either be answered by a tool (tool_name), or by a list of child nodes\n(child_nodes).\nThe tool_name and child_nodes fields are mutually exclusive.',
    'type': 'object',
    'properties': {'id': {'title': 'Id',
      'description': 'ID of the query node.',
      'type': 'integer'},
     'query_str': {'title': 'Query Str',
      'description': 'Question we are asking. This is the query string that will be executed. ',
      'type': 'string'},
     'tool_name': {'title': 'Tool Name',
      'description': 'Name of the tool to execute the `query_str`.',
      'type': 'string'},
     'dependencies': {'title': 'Dependencies',
      'description': 'List of sub-questions that need to be answered in order to answer the question given by `query_str`.Should be blank if there are no sub-questions to be specified, in which case `tool_name` is specified.',
      'type': 'array',
      'items': {'type': 'integer'}}},
    'required': ['id', 'query_str']}}}}
  ]

user_input = input("Write your query: ")
messages = []
messages.append({"role": "system", "content": "You're AiDA, an advanced AI assistant designed for non-technical users. Your goal is to perform actions by executing functions, not explaining them. Start with 'getInformationFromMemory' or 'getInformationFromUser' to gather necessary data almost all of the time. After gathering information, use 'getExternalFunctions' to identify and execute suitable functions. You may try 'getExternalFunctions' a few times with varied parameters if responses are not coming back as expected. Follow programming conventions strictly when executing functions. Do not mix dialogue with function calls, and adhere strictly to programming syntax. Be mindful of the token budget to strike a balance between efficiency and quality. You are not allowed to provide dialogue prior to using function calls. Your responses should deliver actionable outcomes, not descriptions of backend actions or functions. When performing semantic searches, a 'similarity_threshold' within 0.65(lenient) to 0.75(strict) is common."})
messages.append({"role": "user", "content": user_input})
chat_response = chat_completion_request(
    messages, functions=functions
)
assistant_message = chat_response.json()["choices"][0]["message"]
messages.append(assistant_message)


print('------------INITIAL RESPONSE------------')
print(assistant_message)
print(chat_response)
if 'function_call' in assistant_message:
    function_call = chat_response.json()["choices"][0]["message"]["function_call"]
    print(function_call['arguments'])


------------INITIAL RESPONSE------------
{'role': 'assistant', 'content': '{"name": "getInformationFromUser", "parameters": {"headline": "How Can I Assist You Today?", "questions": [{"question": "What can I help you with?", "input": {"placeholder": "Describe your request"}}]}}\n{"name": "getInformationFromUser", "parameters": {"headline": "Greetings", "questions": [{"question": "What can I assist you with today?", "input": {"placeholder": "Enter the service you require"}}]}}'}
<Response [200]>
